# Working with AQuA-RAT and LogiQA Datasets

This notebook demonstrates how to load and inspect samples from the **AQuA-RAT** and **LogiQA** datasets using the dataset classes defined in `src/dataset/aqua.py` and `src/dataset/logiqa.py`.

In [2]:
import sys
sys.path.insert(0, "..")  # make src/ importable from the notebooks/ directory

## 1. LogiQA Dataset

In [2]:
from src.dataset.logiqa import LogiQA_Dataset
from src.configs import DatasetConfig, PromptStyle

# Load the LogiQA test split with a direct (one-word) prompt style
logiqa_config = DatasetConfig(
    path="",                          # unused – data is fetched from GitHub
    prompt_style=PromptStyle.ONE_WORD_NO_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "test"},
)

logiqa = LogiQA_Dataset(logiqa_config)
print(f"Number of test examples: {len(logiqa)}")

Fetching LogiQA 'test' split from GitHub …
LogiQA test: 651 examples loaded.
Number of test examples: 651


### Inspect the raw data for a sample

In [3]:
import pprint

sample_idx = 0

# Raw dict from the parsed dataset
raw_sample = logiqa.data[sample_idx]
print("=== Raw sample ===")
pprint.pprint(raw_sample)

=== Raw sample ===
{'context': 'In the planning of a new district in a township, it was decided '
            'to build a special community in the southeast, northwest, '
            'centered on the citizen park. These four communities are '
            'designated as cultural area, leisure area, commercial area and '
            'administrative service area. It is known that the administrative '
            'service area is southwest of the cultural area, and the cultural '
            'area is southeast of the leisure area.',
 'correct_option': 0,
 'options': ['Civic Park is north of the administrative service area.',
             'The leisure area is southwest of the cultural area.',
             'The cultural district is in the northeast of the business '
             'district.',
             'The business district is southeast of the leisure area.'],
 'query': 'Based on the above statement, which of the following can be '
          'derived?'}


### Build a model-ready prompt and retrieve the answer

In [5]:
# __getitem__ calls build_prompt internally
prompt = logiqa[sample_idx]
print("=== Prompt (ONE_WORD_NO_TAGS, plain string) ===\n")
print(prompt)
print("\n=== answer ===")
print(logiqa.get_correct_letter(sample_idx))

=== Prompt (ONE_WORD_NO_TAGS, plain string) ===

Read the passage and answer the multiple-choice question. Reply with only A, B, C, or D.

Context: In the planning of a new district in a township, it was decided to build a special community in the southeast, northwest, centered on the citizen park. These four communities are designated as cultural area, leisure area, commercial area and administrative service area. It is known that the administrative service area is southwest of the cultural area, and the cultural area is southeast of the leisure area.

Question: Based on the above statement, which of the following can be derived?

Options:
A) Civic Park is north of the administrative service area.
B) The leisure area is southwest of the cultural area.
C) The cultural district is in the northeast of the business district.
D) The business district is southeast of the leisure area.

=== answer ===
A


### Chain-of-thought prompt style

In [6]:
logiqa_cot_config = DatasetConfig(
    path="",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_NO_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "test"},
)

logiqa_cot = LogiQA_Dataset(logiqa_cot_config)
print(logiqa_cot[sample_idx])

Fetching LogiQA 'test' split from GitHub …
LogiQA test: 651 examples loaded.
Read the passage and answer the multiple-choice question. Think step by step, then clearly state your final answer as A, B, C, or D.

Context: In the planning of a new district in a township, it was decided to build a special community in the southeast, northwest, centered on the citizen park. These four communities are designated as cultural area, leisure area, commercial area and administrative service area. It is known that the administrative service area is southwest of the cultural area, and the cultural area is southeast of the leisure area.

Question: Based on the above statement, which of the following can be derived?

Options:
A) Civic Park is north of the administrative service area.
B) The leisure area is southwest of the cultural area.
C) The cultural district is in the northeast of the business district.
D) The business district is southeast of the leisure area.


### Chat-template format

In [7]:
logiqa_chat_config = DatasetConfig(
    path="",
    prompt_style=PromptStyle.ONE_WORD_NO_TAGS,
    use_chat_template=True,          # returns list[dict] instead of a plain string
    hf_data_config={"split": "test"},
)

logiqa_chat = LogiQA_Dataset(logiqa_chat_config)
print("=== Chat-template format ===")
pprint.pprint(logiqa_chat[sample_idx])

Fetching LogiQA 'test' split from GitHub …
LogiQA test: 651 examples loaded.
=== Chat-template format ===
[{'content': 'Read the passage and answer the multiple-choice question. Reply '
             'with only A, B, C, or D.\n'
             '\n'
             'Context: In the planning of a new district in a township, it was '
             'decided to build a special community in the southeast, '
             'northwest, centered on the citizen park. These four communities '
             'are designated as cultural area, leisure area, commercial area '
             'and administrative service area. It is known that the '
             'administrative service area is southwest of the cultural area, '
             'and the cultural area is southeast of the leisure area.\n'
             '\n'
             'Question: Based on the above statement, which of the following '
             'can be derived?\n'
             '\n'
             'Options:\n'
             'A) Civic Park is north of the adm

### Parsing a simulated model response

In [8]:
# Simulate a direct (one-word) model response
direct_response = "B"
parsed = logiqa.parse_model_answer(direct_response)
gold = logiqa.get_correct_letter(sample_idx)
print(f"Model answer : {parsed}")
print(f"Gold answer  : {gold}")
print(f"Correct      : {parsed == gold}")

# Simulate a chain-of-thought response
cot_response = (
    "Looking at the passage, option A is too broad. "
    "After careful reasoning, the answer is B."
)
parsed_cot = logiqa_cot.parse_model_answer(cot_response)
print(f"\nCoT parsed answer: {parsed_cot}")

Model answer : B
Gold answer  : A
Correct      : False

CoT parsed answer: B


---

## 2. AQuA-RAT Dataset

In [9]:
from src.dataset.aqua import AQuA_Dataset

# Load the AQuA-RAT test split
aqua_config = DatasetConfig(
    path="aqua_rat",                  # HuggingFace dataset ID
    prompt_style=PromptStyle.ONE_WORD_NO_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "test"},
)

aqua = AQuA_Dataset(aqua_config)
print(f"Number of test examples: {len(aqua)}")

Data successfully loaded.
Number of test examples: 254


### Inspect the raw data for a sample

In [10]:
raw_aqua = aqua.data[sample_idx]
print("=== Raw sample ===")
pprint.pprint(dict(raw_aqua))

=== Raw sample ===
{'correct': 'A',
 'options': ['A)5(√3 + 1)',
             'B)6(√3 + √2)',
             'C)7(√3 – 1)',
             'D)8(√3 – 2)',
             'E)None of these'],
 'question': 'A car is being driven, in a straight line and at a uniform '
             'speed, towards the base of a vertical tower. The top of the '
             'tower is observed from the car and, in the process, it takes 10 '
             'minutes for the angle of elevation to change from 45° to 60°. '
             'After how much more time will this car reach the base of the '
             'tower?',
 'rationale': 'Explanation :\n'
              'Let the height of the building be h. Initially, he was at an '
              'angle of 450. tan 45 = h/distance between car and tower. h = '
              'distance between car and tower (since tan 45 = 1).\n'
              'Now, after 10 minutes, it travelled a certain distance, and '
              'angle changed to 600.\n'
              'tan 60 = h/x x = h/√

### Build a model-ready prompt and retrieve the gold answer

In [11]:
prompt_aqua = aqua[sample_idx]
print("=== Prompt (ONE_WORD_NO_TAGS, plain string) ===\n")
print(prompt_aqua)
print("\n=== Gold answer letter ===")
print(aqua.get_correct_letter(sample_idx))

=== Prompt (ONE_WORD_NO_TAGS, plain string) ===

Solve the math problem and reply with only A, B, C, D, or E.

Question: A car is being driven, in a straight line and at a uniform speed, towards the base of a vertical tower. The top of the tower is observed from the car and, in the process, it takes 10 minutes for the angle of elevation to change from 45° to 60°. After how much more time will this car reach the base of the tower?

Options:
A) 5(√3 + 1)
B) 6(√3 + √2)
C) 7(√3 – 1)
D) 8(√3 – 2)
E) None of these

=== Gold answer letter ===
A


### Chain-of-thought prompt style

In [12]:
aqua_cot_config = DatasetConfig(
    path="aqua_rat",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_NO_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "test"},
)

aqua_cot = AQuA_Dataset(aqua_cot_config)
print(aqua_cot[sample_idx])

Data successfully loaded.
Solve the math problem step by step, then clearly state your final answer as A, B, C, D, or E.

Question: A car is being driven, in a straight line and at a uniform speed, towards the base of a vertical tower. The top of the tower is observed from the car and, in the process, it takes 10 minutes for the angle of elevation to change from 45° to 60°. After how much more time will this car reach the base of the tower?

Options:
A) 5(√3 + 1)
B) 6(√3 + √2)
C) 7(√3 – 1)
D) 8(√3 – 2)
E) None of these


### Chat-template format

In [13]:
aqua_chat_config = DatasetConfig(
    path="aqua_rat",
    prompt_style=PromptStyle.ONE_WORD_NO_TAGS,
    use_chat_template=True,
    hf_data_config={"split": "test"},
)

aqua_chat = AQuA_Dataset(aqua_chat_config)
print("=== Chat-template format ===")
pprint.pprint(aqua_chat[sample_idx])

Data successfully loaded.
=== Chat-template format ===
[{'content': 'Solve the math problem and reply with only A, B, C, D, or E.\n'
             '\n'
             'Question: A car is being driven, in a straight line and at a '
             'uniform speed, towards the base of a vertical tower. The top of '
             'the tower is observed from the car and, in the process, it takes '
             '10 minutes for the angle of elevation to change from 45° to 60°. '
             'After how much more time will this car reach the base of the '
             'tower?\n'
             '\n'
             'Options:\n'
             'A) 5(√3 + 1)\n'
             'B) 6(√3 + √2)\n'
             'C) 7(√3 – 1)\n'
             'D) 8(√3 – 2)\n'
             'E) None of these',
  'role': 'user'}]


### Parsing a simulated model response

In [14]:
# Simulate a direct (one-word) model response
direct_response = "C"
parsed_aqua = aqua.parse_model_answer(direct_response)
gold_aqua = aqua.get_correct_letter(sample_idx)
print(f"Model answer : {parsed_aqua}")
print(f"Gold answer  : {gold_aqua}")
print(f"Correct      : {parsed_aqua == gold_aqua}")

# Simulate a chain-of-thought response
cot_response = (
    "Let me work through this step by step. "
    "After calculating, the answer is C."
)
parsed_cot_aqua = aqua_cot.parse_model_answer(cot_response)
print(f"\nCoT parsed answer: {parsed_cot_aqua}")

Model answer : C
Gold answer  : A
Correct      : False

CoT parsed answer: C


---

## 3. GSM8K Dataset

In [3]:
from src.dataset.gsm8k import GSM8K_Dataset
from src.configs import DatasetConfig, PromptStyle

# Load the GSM8K test split with a direct (one-word) prompt style
gsm8k_config = DatasetConfig(
    path="openai/gsm8k",
    prompt_style=PromptStyle.ONE_WORD_NO_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "test"},
)

gsm8k = GSM8K_Dataset(gsm8k_config)
print(f"Number of test examples: {len(gsm8k)}")

Loading GSM8K 'test' split from HuggingFace …


README.md: 0.00B [00:00, ?B/s]

GSM8K test: 1319 examples loaded.
Number of test examples: 1319


### Inspect the raw data for a sample

In [4]:
import pprint

sample_idx = 0

# Raw dict straight from HuggingFace – just "question" and "answer"
raw_gsm8k = gsm8k.data[sample_idx]
print("=== Raw sample ===")
pprint.pprint(dict(raw_gsm8k))

=== Raw sample ===
{'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\n'
           'She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n'
           '#### 18',
 'question': 'Janet’s ducks lay 16 eggs per day. She eats three for breakfast '
             'every morning and bakes muffins for her friends every day with '
             "four. She sells the remainder at the farmers' market daily for "
             '$2 per fresh duck egg. How much in dollars does she make every '
             "day at the farmers' market?"}


The `answer` field contains the full chain-of-thought working followed by the gold numeric answer after a `####` marker. `get_correct_answer` extracts just that number.

In [5]:
print("Gold answer:", gsm8k.get_correct_answer(sample_idx))

Gold answer: 18


### Build a model-ready prompt and retrieve the gold answer

In [6]:
prompt = gsm8k[sample_idx]
print("=== Prompt (ONE_WORD_NO_TAGS, plain string) ===\n")
print(prompt)
print("\n=== Gold answer ===")
print(gsm8k.get_correct_answer(sample_idx))

=== Prompt (ONE_WORD_NO_TAGS, plain string) ===

Solve the math problem. Reply with only the final numeric answer, no working.

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

=== Gold answer ===
18


### Chain-of-thought prompt style

In [7]:
gsm8k_cot_config = DatasetConfig(
    path="openai/gsm8k",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_NO_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "test"},
)

gsm8k_cot = GSM8K_Dataset(gsm8k_cot_config)
print(gsm8k_cot[sample_idx])

Loading GSM8K 'test' split from HuggingFace …
GSM8K test: 1319 examples loaded.
Solve the math problem step by step. At the end of your response, write the final numeric answer on its own line in the format: #### <number>

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?


### Chat-template format

In [8]:
gsm8k_chat_config = DatasetConfig(
    path="openai/gsm8k",
    prompt_style=PromptStyle.ONE_WORD_NO_TAGS,
    use_chat_template=True,          # returns list[dict] instead of a plain string
    hf_data_config={"split": "test"},
)

gsm8k_chat = GSM8K_Dataset(gsm8k_chat_config)
print("=== Chat-template format ===")
pprint.pprint(gsm8k_chat[sample_idx])

Loading GSM8K 'test' split from HuggingFace …
GSM8K test: 1319 examples loaded.
=== Chat-template format ===
[{'content': 'Solve the math problem. Reply with only the final numeric '
             'answer, no working.\n'
             '\n'
             'Question: Janet’s ducks lay 16 eggs per day. She eats three for '
             'breakfast every morning and bakes muffins for her friends every '
             "day with four. She sells the remainder at the farmers' market "
             'daily for $2 per fresh duck egg. How much in dollars does she '
             "make every day at the farmers' market?",
  'role': 'user'}]


### Parsing a simulated model response

Unlike the MCQ datasets, `parse_model_answer` extracts a **numeric string** rather than an option letter. The `is_correct` helper handles numeric normalisation so that `"18"` and `"18.0"` are treated as equal.

In [9]:
gold = gsm8k.get_correct_answer(sample_idx)
print(f"Gold answer: {gold}\n")

# Simulate a direct (numeric only) model response
direct_response = gold  # pretend model got it right
parsed = gsm8k.parse_model_answer(direct_response)
print(f"Direct response  : {direct_response!r}")
print(f"Parsed           : {parsed}")
print(f"is_correct       : {gsm8k.is_correct(sample_idx, direct_response)}")

# Simulate a correct CoT response using the #### marker
cot_correct = f"First I calculate the totals step by step...\n#### {gold}"
print(f"\nCoT response     : {cot_correct!r}")
print(f"Parsed           : {gsm8k_cot.parse_model_answer(cot_correct)}")
print(f"is_correct       : {gsm8k_cot.is_correct(sample_idx, cot_correct)}")

# Simulate a wrong response
cot_wrong = "I think the answer is #### 999"
print(f"\nWrong CoT        : {cot_wrong!r}")
print(f"Parsed           : {gsm8k_cot.parse_model_answer(cot_wrong)}")
print(f"is_correct       : {gsm8k_cot.is_correct(sample_idx, cot_wrong)}")

Gold answer: 18

Direct response  : '18'
Parsed           : 18
is_correct       : True

CoT response     : 'First I calculate the totals step by step...\n#### 18'
Parsed           : 18
is_correct       : True

Wrong CoT        : 'I think the answer is #### 999'
Parsed           : 999
is_correct       : False
